# 05 — Divergence Judge

**Owner:** Ian Schmitt (T3) · **Reads:** `outputs/predictions/*.parquet` (from 02/03/04), `data/processed/splits.parquet`, `data/golden/golden_set.csv` · **Writes:** `outputs/tables/05-judge_*.csv`, `outputs/figures/05-judge_*.png`

Where the two classifiers disagree, a locally served LLM adjudicates each review **blind** — no gold label, no model votes — as an independent third opinion. This notebook carries the judge machinery, its frozen configuration, the golden-set evaluation that froze it, and the disagreement adjudication and taxonomy analysis.

Development record: [`../docs/judge-dev-log.md`](../docs/judge-dev-log.md). Annotation guideline: [`../docs/tagging-protocol.md`](../docs/tagging-protocol.md).

**Status:** the judge configuration, the golden-set evaluation, and the disagreement adjudication + baseline scoring on **both** val and test (plan steps 1–4) are live and reproducible. Blind tagging of the two 50-review samples (step 5, via `scripts/tag_disagreements.py`) and the taxonomy analysis (step 6) remain. The notebook always runs clean top to bottom.

In [1]:
# Shared foundation from src/shared.py (landed with 00_core).
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from shared import PATHS, SEED, load_splits

In [2]:
import json
import time

import pandas as pd
from dotenv import dotenv_values
from openai import OpenAI
from scipy.stats import binomtest
from sklearn.metrics import cohen_kappa_score

TBL = PATHS["tables_dir"]
PREDICTIONS = PATHS["predictions_dir"]
GOLDEN_CSV = PATHS["repo_root"] / "data" / "golden" / "golden_set.csv"

## Frozen judge configuration

Ratified 2026-07-23 after the golden-set evaluations (full history in the dev log): **prompt v3**, temperature 0, reasoning disabled, output schema-constrained to a forced binary choice, blind. The endpoint is provider-agnostic: `LLM_BASE_URL` / `LLM_API_KEY` / `LLM_MODEL` come from the gitignored `.env` (`cp .env.example .env`), so the same code targets a local llama.cpp server or any OpenAI-compatible cloud endpoint. Temperature and the no-thinking flag ride in **every request**, so the frozen behavior holds no matter how the serving side was launched.

In [3]:
env = dotenv_values(PATHS["repo_root"] / ".env")
if not env.get("LLM_BASE_URL"):
    raise RuntimeError("No LLM endpoint configured: cp .env.example .env and fill it in.")

client = OpenAI(
    base_url=env["LLM_BASE_URL"], api_key=env["LLM_API_KEY"], timeout=600, max_retries=2
)
MODEL = env["LLM_MODEL"]

JUDGE_PROMPT = (
    "You are a sentiment adjudicator for IMDB movie reviews. Each review's author "
    "also gave the film a star rating from 1 to 10, and the dataset's label is "
    "derived from that rating (>= 7 positive, <= 4 negative). Read the review, "
    "infer the reviewer's verdict, and classify it as positive or negative.\n\n"
    "Judge the reviewer's verdict on the film, not the quality of their writing "
    "and not your own opinion of the film. Where a review is mixed, decide which "
    "way the weighing lands. Where praise or criticism is ironic, the intended "
    "meaning governs, not the surface words. You must choose exactly one label, "
    "even for mixed reviews. Respond in JSON."
)

JUDGE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "verdict",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "label": {"type": "string", "enum": ["positive", "negative"]}
            },
            "required": ["label"],
            "additionalProperties": False,
        },
    },
}


def judge(text):
    """One blind adjudication: review text in, 'positive' or 'negative' out.

    temperature=0 and enable_thinking=false are set per request so the frozen
    config holds on any endpoint; the schema constraint makes malformed output
    impossible by construction.
    """
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": JUDGE_PROMPT},
            {"role": "user", "content": text},
        ],
        response_format=JUDGE_SCHEMA,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return json.loads(resp.choices[0].message.content)["label"]

## Golden-set evaluation

The instrument that selected and froze the configuration: 32 hand-curated IMDB **train**-split reviews (17 negative / 14 positive; 10 easy / 22 hard across negation, mixed, sarcasm, and short/noisy), sourced from the train split so prompt development never touches data that could reach the real adjudication runs. Frozen result: **31/32 overall (96.9%), 21/22 hard (95.5%)**. The single miss, [17596], is the label-noise exemplar — its prose argues against its own gold label — kept in the set deliberately. Re-running this section reproduces the numbers (greedy decoding is deterministic for a fixed server build); with the checkpoint file present it costs zero API calls.

In [4]:
def adjudicate(frame, out_csv, id_col="id", text_col="text", fn=None, label_col="judge_label"):
    """Run an LLM pass over a dataframe, checkpointing one row per result.

    Interrupt-safe: ids already present in out_csv are skipped on re-run, so a
    killed run resumes where it stopped and a completed run costs nothing.
    `fn` defaults to the frozen judge; the v1.2 tag pass reuses the same runner
    with fn=llm_tag and label_col="llm_tag".
    """
    fn = fn or judge
    out_csv = pathlib.Path(out_csv)
    if out_csv.exists():
        done = pd.read_csv(out_csv)
    else:
        done = pd.DataFrame(columns=[id_col, label_col, "latency_s"])
    todo = frame[~frame[id_col].isin(done[id_col])]
    print(f"{len(done)} checkpointed, {len(todo)} to run -> {out_csv.name}")
    for row in todo.itertuples():
        t0 = time.perf_counter()
        label = fn(getattr(row, text_col))
        verdict = {
            id_col: getattr(row, id_col),
            label_col: label,
            "latency_s": round(time.perf_counter() - t0, 2),
        }
        done = pd.concat([done, pd.DataFrame([verdict])], ignore_index=True)
        done.to_csv(out_csv, index=False)
    return done

In [5]:
golden = pd.read_csv(GOLDEN_CSV)
verdicts = adjudicate(
    golden, TBL / "05-judge_golden_eval.csv", id_col="train_idx", text_col="text"
)

ev = golden.merge(verdicts, on="train_idx")
ev["correct"] = ev["judge_label"] == ev["gold"]
hard = ev[ev["difficulty"] == "hard"]
print(
    f"overall {ev['correct'].sum()}/{len(ev)} = {ev['correct'].mean():.1%} | "
    f"hard {hard['correct'].sum()}/{len(hard)} = {hard['correct'].mean():.1%}"
)
for miss in ev[~ev["correct"]].itertuples():
    print(f"miss [{miss.train_idx}] {miss.category}: gold={miss.gold}")

32 checkpointed, 0 to run -> 05-judge_golden_eval.csv
overall 31/32 = 96.9% | hard 21/22 = 95.5%
miss [17596] mixed: gold=positive


## Disagreement adjudication

A disagreement is `y_pred_lr != y_pred_nn`, computed in one merge from the committed prediction files with review text rehydrated from `splits.parquet` (the tagging protocol's Inputs section specifies the blinding). The guard below activates the remaining sections when the prediction files exist.

In [6]:
NEEDED = ["02-lr_val.parquet", "03-nn_val.parquet"]
missing = [name for name in NEEDED if not (PREDICTIONS / name).exists()]
HAVE_PREDICTIONS = not missing
if HAVE_PREDICTIONS:
    print("prediction files present - disagreement sections active")
else:
    print("waiting on:", ", ".join(missing), "(02/03 not merged yet)")

prediction files present - disagreement sections active


In [7]:
def judge_vs_baseline(merged, id_col="id"):
    """Score the judge against the pre-registered baseline on a disagreement set.

    Where the two models disagree, exactly one of them is correct on every row,
    so acc_lr + acc_nn == 1 over the set. The null is therefore not a coin flip
    but "always side with the better model", i.e. max(acc_lr, acc_nn). The judge
    earns its place only by beating that.

    Registered in docs/judge-dev-log.md (2026-07-24) and committed before any
    prediction file existed, so the bar cannot be chosen after seeing the data.

    `merged` carries y_true, y_pred_lr, y_pred_nn and judge_label per id.
    """
    truth = merged["y_true"]
    acc_lr = float((merged["y_pred_lr"] == truth).mean())
    acc_nn = float((merged["y_pred_nn"] == truth).mean())
    judged = merged["judge_label"].map({"negative": 0, "positive": 1})
    acc_judge = float((judged == truth).mean())
    baseline = max(acc_lr, acc_nn)
    return {
        "n": len(merged),
        "acc_lr": acc_lr,
        "acc_nn": acc_nn,
        "baseline_best_model": baseline,
        "acc_judge": acc_judge,
        "beats_baseline": acc_judge > baseline,
    }


# Sanity-check the arithmetic on a synthetic frame, so the function is exercised
# even while the real prediction files are still missing.
_probe = pd.DataFrame(
    {
        "y_true": [1, 1, 0, 0],
        "y_pred_lr": [1, 0, 0, 1],
        "y_pred_nn": [0, 1, 1, 0],
        "judge_label": ["positive", "positive", "negative", "negative"],
    }
)
_check = judge_vs_baseline(_probe)
assert _check["acc_lr"] + _check["acc_nn"] == 1.0, "disagreement identity violated"
print("baseline scorer ready:", _check)

baseline scorer ready: {'n': 4, 'acc_lr': 0.5, 'acc_nn': 0.5, 'baseline_best_model': 0.5, 'acc_judge': 1.0, 'beats_baseline': True}


### Planned once predictions land

1. **Derive the val disagreement set** (one merge) and apply the protocol's sampling rule: all if ≤50, else a seeded random 50.
2. **Screen for golden-set overlap.** The golden set is train-sourced and `val` is drawn from the same pool, so up to 8 golden reviews can reappear in a val disagreement set (`source_index` in `splits.parquet` makes this a one-line check). Report the count and exclude if nonzero. The test run needs no such screen: **zero** golden reviews have a text twin in the test split, verified 2026-07-24.
3. **Adjudicate** with `adjudicate()` — the same checkpointed runner exercised above.
4. **Score against the pre-registered baseline** using `judge_vs_baseline()` above: the judge must beat `max(acc_lr, acc_nn)`, not 50%. Reported either way.
5. **Tag** the sampled disagreements blind per the protocol; votes, gold, and judge verdicts rejoin only after tags are recorded.
6. **Analyze:** taxonomy distribution, per-category LR/NN win rates, judge accuracy by category, and the who-was-right readout; repeated on test disagreements after the single final run.

### Val disagreement set — steps 1 and 2

One merge derives the disagreement set; the golden screen and the protocol's sampling rule follow. Nothing here calls the judge — steps 3–5 run against this derivation in a later session — and nothing is stored: disagreements are computed, never written to disk, per the handoff decision in `docs/decisions.md`, so this section re-derives in milliseconds on every run.

In [8]:
if HAVE_PREDICTIONS:
    lr = pd.read_parquet(PREDICTIONS / "02-lr_val.parquet")
    nn = pd.read_parquet(PREDICTIONS / "03-nn_val.parquet")
    merged = lr.merge(nn, on="id", suffixes=("_lr", "_nn"), validate="one_to_one")
    assert len(merged) == len(lr) == len(nn), "prediction files must cover identical ids"
    assert (merged["y_true_lr"] == merged["y_true_nn"]).all(), "y_true disagrees between files"
    merged = merged.rename(columns={"y_true_lr": "y_true"}).drop(columns="y_true_nn")

    disagree = merged.loc[merged["y_pred_lr"] != merged["y_pred_nn"]].copy()
    acc_lr = float((disagree["y_pred_lr"] == disagree["y_true"]).mean())
    acc_nn = float((disagree["y_pred_nn"] == disagree["y_true"]).mean())
    assert abs(acc_lr + acc_nn - 1.0) < 1e-9, "disagreement identity violated"
    print(f"val disagreements: {len(disagree)} of {len(merged)} ({len(disagree) / len(merged):.1%})")
    print(f"on disagreements: acc_lr {acc_lr:.4f} | acc_nn {acc_nn:.4f}")
    print(f"pre-registered baseline max(acc_lr, acc_nn): {max(acc_lr, acc_nn):.4f}")
else:
    print("skipped - waiting on prediction files")

val disagreements: 109 of 5000 (2.2%)
on disagreements: acc_lr 0.3853 | acc_nn 0.6147
pre-registered baseline max(acc_lr, acc_nn): 0.6147


In [9]:
if HAVE_PREDICTIONS:
    splits = load_splits()
    val_rows = splits.loc[splits["split"] == "val", ["id", "source_split", "source_index"]]
    disagree = disagree.merge(val_rows, on="id", validate="one_to_one")

    golden_idx = set(pd.read_csv(GOLDEN_CSV)["train_idx"])
    is_golden = (disagree["source_split"] == "train") & disagree["source_index"].isin(golden_idx)
    print(f"golden-set reviews in the disagreement set: {int(is_golden.sum())}")
    for row in disagree.loc[is_golden].itertuples():
        print(f"  excluding {row.id} (train_idx {row.source_index})")
    screened = disagree.loc[~is_golden].copy()

    # Protocol sampling rule (pre-registered): tag all if <=50, else a seeded random 50.
    tag_sample = screened if len(screened) <= 50 else screened.sample(n=50, random_state=SEED)
    print(f"screened disagreement set: {len(screened)} | tagging sample: {len(tag_sample)}")
else:
    print("skipped - waiting on prediction files")

golden-set reviews in the disagreement set: 0
screened disagreement set: 109 | tagging sample: 50


### Adjudication and baseline scoring — steps 3 and 4

The full screened set is adjudicated, blind — the judge receives review text only, rehydrated from `splits.parquet`; the 50-row sample above bounds only the *manual* tagging in step 5. Verdicts checkpoint one row at a time to `05-judge_val_disagreements.csv`, so an interrupted run resumes where it stopped and a completed run costs zero API calls. Scoring uses the pre-registered `judge_vs_baseline()` defined above; the result is reported pass or fail.

In [10]:
if HAVE_PREDICTIONS:
    texts = splits.loc[splits["split"] == "val", ["id", "text"]]
    to_judge = screened.merge(texts, on="id", validate="one_to_one")
    verdicts = adjudicate(to_judge, TBL / "05-judge_val_disagreements.csv")
else:
    print("skipped - waiting on prediction files")

109 checkpointed, 0 to run -> 05-judge_val_disagreements.csv


In [11]:
if HAVE_PREDICTIONS:
    scored = screened.merge(verdicts[["id", "judge_label"]], on="id", validate="one_to_one")
    result = judge_vs_baseline(scored)
    for key, value in result.items():
        print(f"{key:>20}: {value}")
    sided_nn = float(
        (scored["judge_label"].map({"negative": 0, "positive": 1}) == scored["y_pred_nn"]).mean()
    )
    print(f"{'judge sided with NN':>20}: {sided_nn:.1%} of rows")
else:
    print("skipped - waiting on prediction files")

                   n: 109
              acc_lr: 0.3853211009174312
              acc_nn: 0.6146788990825688
 baseline_best_model: 0.6146788990825688
           acc_judge: 0.908256880733945
      beats_baseline: True
 judge sided with NN: 63.3% of rows


## Test-set adjudication — the headline run

Notebook 04's single final run landed `04-test_predictions.parquet` (long format, one row per id × model), so this section runs the registered plan on the **test** disagreements. No golden screen is needed here — zero golden reviews have a text twin anywhere in the test split (verified 2026-07-24, dev log). The judge enters exactly as frozen on 2026-07-23, and the pre-registered baseline is recomputed on this set. The headline is reported with a Wilson 95% interval and an exact binomial test against the side-with-the-better-model null.

In [12]:
TEST_PRED = PREDICTIONS / "04-test_predictions.parquet"
HAVE_TEST = TEST_PRED.exists()
if HAVE_TEST:
    tp = pd.read_parquet(TEST_PRED)
    lr_t = tp.loc[tp["model"] == "logistic_regression", ["id", "y_true", "y_pred"]]
    nn_t = tp.loc[tp["model"] == "neural_network", ["id", "y_true", "y_pred"]]
    test = lr_t.merge(nn_t, on="id", suffixes=("_lr", "_nn"), validate="one_to_one")
    assert (test["y_true_lr"] == test["y_true_nn"]).all(), "y_true disagrees between models"
    test = test.rename(columns={"y_true_lr": "y_true"}).drop(columns="y_true_nn")

    test_dis = test.loc[test["y_pred_lr"] != test["y_pred_nn"]].copy()
    t_acc_lr = float((test_dis["y_pred_lr"] == test_dis["y_true"]).mean())
    t_acc_nn = float((test_dis["y_pred_nn"] == test_dis["y_true"]).mean())
    assert abs(t_acc_lr + t_acc_nn - 1.0) < 1e-9, "disagreement identity violated"
    test_tag_sample = (
        test_dis if len(test_dis) <= 50 else test_dis.sample(n=50, random_state=SEED)
    )
    print(f"test disagreements: {len(test_dis)} of {len(test)} ({len(test_dis) / len(test):.2%})")
    print(f"on disagreements: acc_lr {t_acc_lr:.4f} | acc_nn {t_acc_nn:.4f}")
    print(f"pre-registered baseline max(acc_lr, acc_nn): {max(t_acc_lr, t_acc_nn):.4f}")
    print(f"tagging sample (protocol rule): {len(test_tag_sample)}")
else:
    print("waiting on 04-test_predictions.parquet (04 not merged yet)")

test disagreements: 693 of 25000 (2.77%)
on disagreements: acc_lr 0.4127 | acc_nn 0.5873
pre-registered baseline max(acc_lr, acc_nn): 0.5873
tagging sample (protocol rule): 50


In [13]:
if HAVE_TEST:
    test_texts = splits.loc[splits["split"] == "test", ["id", "text"]]
    to_judge_test = test_dis.merge(test_texts, on="id", validate="one_to_one")
    test_verdicts = adjudicate(to_judge_test, TBL / "05-judge_test_disagreements.csv")
else:
    print("skipped - waiting on 04-test_predictions.parquet")

693 checkpointed, 0 to run -> 05-judge_test_disagreements.csv


In [14]:
if HAVE_TEST:
    test_scored = test_dis.merge(
        test_verdicts[["id", "judge_label"]], on="id", validate="one_to_one"
    )
    test_result = judge_vs_baseline(test_scored)
    for key, value in test_result.items():
        print(f"{key:>20}: {value}")
    sided_nn_test = float(
        (test_scored["judge_label"].map({"negative": 0, "positive": 1})
         == test_scored["y_pred_nn"]).mean()
    )
    print(f"{'judge sided with NN':>20}: {sided_nn_test:.1%} of rows")

    # Uncertainty on the headline: Wilson 95% interval, and an exact binomial
    # test against the pre-registered side-with-the-better-model null.
    k = int(round(test_result["acc_judge"] * test_result["n"]))
    bt = binomtest(k, test_result["n"], test_result["baseline_best_model"], alternative="greater")
    ci = binomtest(k, test_result["n"]).proportion_ci(confidence_level=0.95, method="wilson")
    print(f"{'Wilson 95% CI':>20}: [{ci.low:.4f}, {ci.high:.4f}]")
    print(f"{'exact binomial p':>20}: {bt.pvalue:.2e} (vs baseline null)")
else:
    print("skipped - waiting on 04-test_predictions.parquet")

                   n: 693
              acc_lr: 0.4126984126984127
              acc_nn: 0.5873015873015873
 baseline_best_model: 0.5873015873015873
           acc_judge: 0.9278499278499278
      beats_baseline: True
 judge sided with NN: 57.3% of rows
       Wilson 95% CI: [0.9061, 0.9448]
    exact binomial p: 9.48e-92 (vs baseline null)


## LLM tag pass and human agreement — protocol v1.2

Amendment v1.2 (adopted 2026-08-02, mid-val-session, before any test tagging — see `../docs/tagging-protocol.md`): a **separate** LLM pass tags **every** disagreement row on both splits with the protocol's five categories, using the protocol's own decision rules as the prompt, temperature 0, schema-forced choice, blind to gold labels, model votes, and judge verdicts. The frozen verdict run is never regenerated. The human val sample (tagged cold under v1.1) doubles as the anchor-free reference: raw agreement, Cohen's κ, and the annotator confusion matrix below. Reporting: the human(-verified) 50s are the primary taxonomy; the LLM-tagged full sets (109 + 693) are the secondary, full-coverage readout.

In [15]:
TAG_PROMPT = (
    "You are annotating IMDB movie reviews for a study of why two text classifiers "
    "disagree. Assign exactly ONE category describing the text property that most "
    "plausibly makes the review hard to classify. Apply these tests in precedence "
    "order - noise > sarcasm > negation > mixed > other - and choose the first that "
    "clearly applies.\n\n"
    "noise: the text is defective as sentiment evidence - wrong language, dominated "
    "by markup junk, or an opinion-free plot synopsis. Test: a careful reader could "
    "not argue ANY verdict from this text. Broken prose with clear polarity is NOT "
    "noise.\n\n"
    "sarcasm: inversion - surface-positive vocabulary carries a negative verdict (or "
    "vice versa): ironic superlatives, mock praise, rhetorical disbelief, "
    "so-bad-it's-good enjoyment. Plain insult or contempt in surface-negative words "
    "does not invert and is NOT sarcasm.\n\n"
    "negation: the overall sentiment is carried by negation or reversal structure, "
    "judged on the WHOLE review - praise phrased through negatives, polite-negative "
    "pans, hope-then-disappointment arcs where praise vocabulary aims at expectations "
    "rather than the film. Misreading the negation would flip the review's polarity. "
    "A negation phrase inside an otherwise plainly polarized review does NOT "
    "qualify.\n\n"
    "mixed: genuinely two-sided - substantial praise AND substantial criticism both "
    "carry real weight, and the verdict rests on their balance. A one-sided review "
    "with minor concessions is NOT mixed. If the balance is itself carried by "
    "stacked negations, choose negation.\n\n"
    "other: none of the above clearly applies - ordinary assessable sentiment, "
    "including verdicts diluted by long plot synopsis or topic vocabulary.\n\n"
    "Respond in JSON."
)

TAG_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "taxonomy_tag",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "tag": {
                    "type": "string",
                    "enum": ["negation", "sarcasm", "mixed", "noise", "other"],
                }
            },
            "required": ["tag"],
            "additionalProperties": False,
        },
    },
}


def llm_tag(text):
    """One blind taxonomy tag: review text in, one of five categories out.

    A separate pass from the verdict run (protocol v1.2): the frozen judge
    configuration and its checkpointed verdicts are never regenerated here,
    and this pass sees the same input the human tagger sees - text only.
    """
    resp = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": TAG_PROMPT},
            {"role": "user", "content": text},
        ],
        response_format=TAG_SCHEMA,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return json.loads(resp.choices[0].message.content)["tag"]

In [16]:
if HAVE_PREDICTIONS:
    val_llm_tags = adjudicate(
        to_judge, TBL / "05-judge_val_llm_tags.csv", fn=llm_tag, label_col="llm_tag"
    )
if HAVE_TEST:
    test_llm_tags = adjudicate(
        to_judge_test, TBL / "05-judge_test_llm_tags.csv", fn=llm_tag, label_col="llm_tag"
    )

109 checkpointed, 0 to run -> 05-judge_val_llm_tags.csv
693 checkpointed, 0 to run -> 05-judge_test_llm_tags.csv


In [17]:
HUMAN_VAL = TBL / "05-judge_val_tags.csv"
if HAVE_PREDICTIONS and HUMAN_VAL.exists():
    # the harness writes an llm_tag column of its own (verify mode); drop it here
    human = pd.read_csv(HUMAN_VAL).drop(columns=["llm_tag"], errors="ignore")
    both = human.merge(val_llm_tags[["id", "llm_tag"]], on="id", validate="one_to_one")
    agree = float((both["tag"] == both["llm_tag"]).mean())
    kappa = cohen_kappa_score(both["tag"], both["llm_tag"])
    print(f"human-LLM agreement on {len(both)} val tags: {agree:.1%} | Cohen's kappa {kappa:.3f}")
    print(pd.crosstab(both["tag"], both["llm_tag"], rownames=["human"], colnames=["llm"]))
else:
    print("human val tags not present yet - agreement runs once 05-judge_val_tags.csv exists")

human-LLM agreement on 50 val tags: 32.0% | Cohen's kappa 0.150
llm       mixed  negation  noise  other  sarcasm
human                                           
mixed         6         0      1      0        0
negation      3         1      0      1        0
other        18         1      4      7        2
sarcasm       3         0      1      0        2


In [18]:
# Handoff manifest: every artifact this notebook wrote, for the record.
written = sorted(
    p.relative_to(PATHS["repo_root"]) for p in TBL.glob("05-judge_*.csv")
)
print(f"{len(written)} artifacts written:")
for p in written:
    print(" ", p)

6 artifacts written:
  outputs/tables/05-judge_golden_eval.csv
  outputs/tables/05-judge_test_disagreements.csv
  outputs/tables/05-judge_test_llm_tags.csv
  outputs/tables/05-judge_val_disagreements.csv
  outputs/tables/05-judge_val_llm_tags.csv
  outputs/tables/05-judge_val_tags.csv
